
# End-to-End Text Fine-Tuning with Granite 4.1 3B Base: LoRA + QLoRA

## Project Overview

This notebook provides a **beginner-friendly, production-style** walkthrough of parameter-efficient fine-tuning (PEFT) on a 3-billion-parameter causal language model — from raw dataset to saved, reloadable adapters.

**Model**: `ibm-granite/granite-4.1-3b-base` — IBM Granite 4.1, a decoder-only causal LM  
**Task**: 4-class news article classification (AG News dataset)  
**Hardware target**: Single consumer GPU (8 GB VRAM), e.g., RTX 3060 / 4060 / 4070

### What You Will Build

| Stage | What Runs | Key Concept |
|-------|-----------|-------------|
| **Base** | Zero-shot evaluation, no training | Measuring the pre-trained model's inherent ability |
| **LoRA** | Fine-tune ~31M adapter params on frozen 3B base | Low-Rank Adaptation on bf16 weights |
| **QLoRA** | Same adapters, but base model in 4-bit NF4 | Quantized base → ~50% less VRAM |
| **Final** | Reload saved adapters, compare all three | Deployment-style adapter loading |

### Learning Objectives

After completing this notebook you will be able to:

1. Explain why full fine-tuning is impractical on consumer hardware and how PEFT solves it.
2. Implement LoRA with HuggingFace PEFT from scratch.
3. Implement QLoRA with BitsAndBytes 4-bit NF4 quantization.
4. Write a generation-based evaluation loop for a causal LM classifier.
5. Save, reload, and optionally merge adapters for deployment.
6. Profile GPU memory usage across training stages.



## Problem Statement

### Why Not Fine-Tune the Full Model?

A 3B-parameter model in `bfloat16` uses **~6 GB of VRAM for weights alone**. Full fine-tuning also needs:

| Component | Memory |
|-----------|--------|
| Weights (bf16) | ~6 GB |
| Gradients (same dtype) | ~6 GB |
| AdamW optimizer states (fp32) | ~12 GB |
| Activations (variable) | ~4–8 GB |
| **Total** | **~28–32 GB** |

This is unachievable on a single 8 GB consumer GPU.

### How PEFT Solves This

**LoRA (Low-Rank Adaptation):**
- Freezes ALL base weights → zero gradient memory for them
- Injects tiny low-rank adapter matrices (A, B) into selected layers
- Only the adapters have gradients → optimizer states shrink from ~12 GB to ~1.2 GB

**QLoRA (Quantized LoRA):**
- Same adapter-only training as LoRA
- Additionally loads the frozen base in **4-bit NF4** quantization → base drops from ~6 GB to ~1.7 GB
- Total training memory under **6 GB** for a 3B model

### This Notebook's Hardware Profile

| Stage | Peak VRAM (estimated) |
|-------|-----------------------|
| Base eval (bf16) | ~6 GB |
| LoRA training (bf16 base + adapters) | ~7–7.5 GB |
| QLoRA training (4-bit base + adapters) | ~4–5 GB |

---

## Environment Setup & Prerequisites

### Hardware Requirements

| Setup | Supported |
|-------|-----------|
| NVIDIA GPU ≥ 8 GB VRAM | ✅ Recommended |
| NVIDIA GPU ≥ 16 GB VRAM | ✅ Comfortable |
| CPU only | ⚠️ Functional but slow (hours per stage) |
| Apple Silicon (MPS) | ⚠️ Base/LoRA only — QLoRA unsupported |

### Create the Environment

```bash
cd /home/ahmad/AI/Notebooks/LoRA-QLoRA
uv venv .venv --python 3.12.10
source .venv/bin/activate
uv pip install \
    "torch>=2.4" "transformers>=4.57" "datasets>=3.0" "peft>=0.17" \
    "accelerate>=1.0" "bitsandbytes>=0.48" "evaluate>=0.4" \
    "scikit-learn>=1.5" "pandas>=2.2" "psutil>=6.0" \
    "ipykernel>=6.29" "jupyter>=1.1"
```

### Register the Jupyter Kernel

```bash
source .venv/bin/activate
python -m ipykernel install --user \
    --name lora-qlora-py312 \
    --display-name "Granite LoRA/QLoRA (py3.12)"
```



## Core Concepts: Transformers, PEFT, LoRA, and QLoRA

### 1. Transformers

Transformers are neural network architectures built on **self-attention**: every token in the input can attend to every other token, capturing long-range dependencies. Modern LLMs are **decoder-only** transformers — they predict the next token autoregressively and are the standard backbone for text generation.

Key layers in each transformer block:
- **Attention** (Q, K, V, O projections): captures contextual relationships
- **MLP/FFN** (up, gate, down projections): applies nonlinear transformations
- **Layer norms**: stabilize training

### 2. PEFT — Parameter-Efficient Fine-Tuning

PEFT is a family of techniques that adapt a pre-trained model for a new task by training only a small number of additional parameters while keeping the base model frozen. Benefits:

- **Memory**: no gradients or optimizer states for frozen weights
- **Storage**: adapter checkpoints are tiny (MB, not GB)
- **Modularity**: swap adapters without touching the base model

### 3. LoRA — Low-Rank Adaptation

LoRA injects two small matrices **A** (shape `[r, d_in]`) and **B** (shape `[d_out, r]`) alongside each target weight matrix **W** (shape `[d_out, d_in]`). During the forward pass:

```
output = W·x + (B·A)·x · (alpha / r)
       = W·x + ΔW·x
```

- **`r` (rank)**: controls adapter capacity. Typical values: 8–64. Higher → more parameters, more expressive.
- **`alpha`**: scaling factor; `lora_alpha / r` controls how strongly the adapter influences the output.
- **`target_modules`**: which layers receive adapters (`"all-linear"` targets every linear layer).

With `r=16` on a 3B model, LoRA adds ~31M trainable parameters — about **0.9%** of total weights.

### 4. QLoRA — Quantized LoRA

QLoRA (Dettmers et al., 2023) combines LoRA with 4-bit NF4 (Normal Float 4) quantization:

1. **4-bit NF4 quantization**: the frozen base weights are stored in a 4-bit data type optimized for normally-distributed weights (typical of pre-trained LLMs). Information loss is minimal.
2. **Double quantization**: the quantization constants themselves are also quantized, saving another ~0.37 bits per parameter.
3. **Paged optimizers**: allow optimizer states to spill to CPU RAM if needed (handled by `bitsandbytes`).
4. **BF16 compute**: despite 4-bit storage, computations are done in bfloat16 for numeric stability.

**Key insight**: QLoRA never materializes the full-precision base model in GPU memory. The base exists only in 4-bit storage; it is dequantized tile-by-tile during each forward pass.

### 5. Quantization Explained

Quantization maps floating-point numbers to a smaller set of representable values:

| Format | Bits | VRAM for 3B params | Notes |
|--------|------|--------------------|-------|
| float32 | 32 | ~12 GB | Training standard |
| bfloat16 | 16 | ~6 GB | BF16 preferred for LLMs (wider range than FP16) |
| int8 | 8 | ~3 GB | Minor quality loss |
| NF4 | 4 | ~1.7 GB | QLoRA standard — designed for normally distributed weights |

**NF4 specifics**: Unlike uniform quantization (equal spacing between representable values), NF4 allocates more values near zero (where most weights cluster) and fewer at the extremes. This minimizes quantization error for normally distributed weight distributions.

### 6. LoRA vs QLoRA — When to Use Which

| Criterion | LoRA | QLoRA |
|-----------|------|-------|
| GPU VRAM | ~7–8 GB for 3B | ~4–5 GB for 3B |
| Training speed | Faster (no dequant overhead) | ~10–30% slower |
| Accuracy | Slightly higher | Comparable, minor degradation |
| Use case | GPU with enough VRAM | Tight VRAM budget |


In [ ]:

# Core imports and version visibility for reproducibility.
import gc
import inspect
import os
import random
import re
import time
from pathlib import Path

# Set CUDA allocator config before any CUDA operations are performed.
# expandable_segments reduces fragmentation, allowing VRAM to be reclaimed
# more aggressively between pipeline stages.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import psutil
import torch
from datasets import Dataset, load_dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)

try:
    import evaluate
except Exception:
    evaluate = None

print(f"torch={torch.__version__}")
print(f"transformers={__import__('transformers').__version__}")
print(f"datasets={__import__('datasets').__version__}")
print(f"peft={__import__('peft').__version__}")
print(f"accelerate={__import__('accelerate').__version__}")
print(f"bitsandbytes={__import__('bitsandbytes').__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Total VRAM: {total_vram:.2f} GB")


In [ ]:

# Global configuration for this tutorial.
SEED = 42
MODEL_ID = "ibm-granite/granite-4.1-3b-base"
PROJECT_ROOT = Path("/home/ahmad/AI/Notebooks/LoRA-QLoRA")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
LORA_ADAPTER_DIR = ARTIFACTS_DIR / "lora_adapter"
QLORA_ADAPTER_DIR = ARTIFACTS_DIR / "qlora_adapter"

# Enable CUDA expandable segments to reduce fragmentation and allow more aggressive
# memory reclaiming between stages. Must be set before any CUDA allocations.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


def parse_env_bool(name: str, default: bool) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


VALID_RUN_STAGES = {"all", "base", "lora", "qlora", "final"}
RUN_STAGE = os.getenv("RUN_STAGE", "all").strip().lower()
if RUN_STAGE not in VALID_RUN_STAGES:
    raise ValueError(f"RUN_STAGE must be one of {sorted(VALID_RUN_STAGES)}, got: {RUN_STAGE}")

FAIL_ON_STAGE_ERROR = parse_env_bool("FAIL_ON_STAGE_ERROR", True)

# Data sizing controls.
# Keep USE_FULL_TRAIN = True for a real, non-toy training run.
USE_FULL_TRAIN = True
TRAIN_MAX_SAMPLES = None if USE_FULL_TRAIN else 20_000
VAL_MAX_SAMPLES = 10_000
GEN_EVAL_MAX_SAMPLES = 2_000  # generation-based evaluation set size

# Tokenization / generation settings.
MAX_SEQ_LENGTH = 384
MAX_SEQ_LENGTH_OVERRIDE = os.getenv("MAX_SEQ_LENGTH_OVERRIDE")
if MAX_SEQ_LENGTH_OVERRIDE:
    MAX_SEQ_LENGTH = int(MAX_SEQ_LENGTH_OVERRIDE)
MAX_NEW_TOKENS_LABEL = 8

# Training settings (tuned for single-GPU LoRA-style training).
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16
LOGGING_STEPS = 50

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
LORA_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
QLORA_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Artifacts dir: {ARTIFACTS_DIR}")
print(f"Run stage: {RUN_STAGE}")
print(f"Fail on stage error: {FAIL_ON_STAGE_ERROR}")
print(f"MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH}")
print(f"PYTORCH_CUDA_ALLOC_CONF: {os.environ.get('PYTORCH_CUDA_ALLOC_CONF', 'not set')}")


In [3]:

# Utility helpers: seeding, runtime detection, memory tracking, and cleanup.

STAGE_EXECUTION_MAP = {
    "all": {"base", "lora", "qlora", "final"},
    "base": {"base"},
    "lora": {"lora"},
    "qlora": {"qlora"},
    "final": {"final"},
}
ACTIVE_STAGES = STAGE_EXECUTION_MAP[RUN_STAGE]


def should_run_stage(stage_name: str) -> bool:
    return stage_name in ACTIVE_STAGES


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_compute_dtype() -> torch.dtype:
    if not torch.cuda.is_available():
        return torch.float32
    if torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


def detect_runtime() -> dict:
    info = {
        "cuda_available": torch.cuda.is_available(),
        "cuda_device_count": torch.cuda.device_count(),
        "compute_dtype": str(get_compute_dtype()),
        "qlora_supported": False,
        "notes": [],
    }
    if info["cuda_available"]:
        info["gpu_name"] = torch.cuda.get_device_name(0)
        info["cuda_version"] = torch.version.cuda
        info["qlora_supported"] = True
    else:
        info["notes"].append("CUDA not detected. QLoRA will be skipped.")

    try:
        import bitsandbytes as bnb  # noqa: F401
    except Exception as e:
        info["qlora_supported"] = False
        info["notes"].append(f"bitsandbytes import failed: {e}")

    return info


def memory_snapshot(tag: str) -> dict:
    process = psutil.Process(os.getpid())
    snapshot = {
        "tag": tag,
        "cpu_rss_gb": round(process.memory_info().rss / (1024 ** 3), 3),
        "time": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    if torch.cuda.is_available():
        snapshot.update(
            {
                "gpu_allocated_gb": round(torch.cuda.memory_allocated() / (1024 ** 3), 3),
                "gpu_reserved_gb": round(torch.cuda.memory_reserved() / (1024 ** 3), 3),
                "gpu_max_allocated_gb": round(torch.cuda.max_memory_allocated() / (1024 ** 3), 3),
                "gpu_max_reserved_gb": round(torch.cuda.max_memory_reserved() / (1024 ** 3), 3),
            }
        )
    return snapshot


def cleanup_torch_objects(*objs) -> None:
    for obj in objs:
        try:
            del obj
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


set_seed(SEED)
runtime = detect_runtime()
runtime


{'cuda_available': True,
 'cuda_device_count': 1,
 'compute_dtype': 'torch.bfloat16',
 'qlora_supported': True,
 'notes': [],
 'gpu_name': 'NVIDIA GeForce RTX 4060 Laptop GPU',
 'cuda_version': '13.0'}


## Step 1: Dataset — AG News Text Classification

### Dataset Description

**AG News** is a news article collection commonly used as an NLP benchmark. We use it here for **4-class text classification**:

| Label ID | Label Name | Example Topic |
|----------|------------|---------------|
| 0 | World | International politics, diplomacy |
| 1 | Sports | Games, athletes, tournaments |
| 2 | Business | Markets, companies, economics |
| 3 | Sci/Tech | Science, software, hardware |

**Dataset statistics:**

| Split | Samples |
|-------|---------|
| Train (original) | 120,000 |
| Train (after val split) | 110,000 |
| Validation (held-out) | 10,000 |
| Test | 7,600 |
| Generation eval (from test) | 2,000 |

**Class balance**: approximately uniform — 25% per class. This means accuracy is a meaningful metric and macro-F1 ≈ accuracy for well-calibrated models.

### Why AG News for This Tutorial?

- Short articles (30–200 tokens) → fits in 384-token max sequence length
- Balanced classes → fair comparison between methods
- Challenging enough to show LoRA/QLoRA improvement over base
- Fast to tokenize and train on (even 110K samples)

### Data Pipeline

```
raw HF dataset
     │
     ├── train (120K) ──► train_test_split(10K val) ──► train_ds (110K)
     │                                               └─► val_ds (10K)
     └── test (7.6K) ──► gen_eval_ds (first 2K for generation eval)
```

The val split is used for **token-level loss monitoring** during training.  
The gen_eval split is used for **generation-based accuracy** (the real metric).


In [4]:
# Load AG News and create train/validation/test splits.
# We try a namespaced ID first because some modern datasets/hub combinations
# require `namespace/name` instead of short aliases.
DATASET_CANDIDATES = [
    "fancyzhx/ag_news",
    "ag_news",
]

raw = None
last_error = None
for ds_id in DATASET_CANDIDATES:
    try:
        raw = load_dataset(ds_id)
        print(f"Loaded dataset: {ds_id}")
        break
    except Exception as e:
        last_error = e
        print(f"Failed loading {ds_id}: {e}")

if raw is None:
    raise RuntimeError(
        "Could not load AG News from any known dataset ID. "
        f"Last error: {last_error}"
    )

print(raw)

train_full = raw["train"]
test_full = raw["test"]

# Validation split from training data.
train_val = train_full.train_test_split(test_size=10_000, seed=SEED, shuffle=True)
train_ds = train_val["train"]
val_ds = train_val["test"]

# Optionally reduce sample counts for debugging (not default).
if TRAIN_MAX_SAMPLES is not None:
    train_ds = train_ds.select(range(min(TRAIN_MAX_SAMPLES, len(train_ds))))
if VAL_MAX_SAMPLES is not None:
    val_ds = val_ds.select(range(min(VAL_MAX_SAMPLES, len(val_ds))))

gen_eval_ds = test_full
if GEN_EVAL_MAX_SAMPLES is not None:
    gen_eval_ds = gen_eval_ds.select(range(min(GEN_EVAL_MAX_SAMPLES, len(gen_eval_ds))))

print(f"Train samples: {len(train_ds):,}")
print(f"Validation samples: {len(val_ds):,}")
print(f"Generation eval samples: {len(gen_eval_ds):,}")

pd.DataFrame(train_ds[:3])

Loaded dataset: fancyzhx/ag_news
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})
Train samples: 110,000
Validation samples: 10,000
Generation eval samples: 2,000


,text,label
0,Congress closes in on reviving Net tax ban WAS...,2
1,Astros turn to Oswalt for crucial game SAN FRA...,1
2,Music industry opens new front against Kazaa i...,3



## Step 2: Prompt Engineering for Causal LM Classification

### Why Prompting for Classification?

Granite 4.1 3B Base is a **causal language model** — it generates the next token, not a class logit. To use it for classification:

1. **Wrap** the input in an instruction-style prompt
2. **Train** (or zero-shot test) by asking it to generate the label word
3. **Parse** the generated text back to a class index

This approach is called **generative classification** or **conditional generation for classification**. It keeps the pipeline identical between base evaluation and fine-tuning, making comparisons fair.

### The Prompt Template

```
Classify the following news article into exactly one label:
World, Sports, Business, or Sci/Tech.
Return only the label.

Article: {article_text}
Label: {label}   ← model generates this
```

### Supervised Training with Label-Only Loss

During training we want the model to **only learn the label token**, not re-learn the prompt text. We achieve this by:

- Setting `labels = [-100, -100, ..., label_token_ids]`
- PyTorch's `CrossEntropyLoss` ignores positions with `label == -100`
- Only the label tokens (1–3 tokens) contribute to the training loss

This is called **prompt masking** or **instruction masking**. Without it, the model would waste capacity learning the fixed instruction template.

### Token Length Analysis

A typical AG News article in this dataset is 30–120 tokens. Our prompt adds ~30 tokens of instruction. With `MAX_SEQ_LENGTH=384`:
- ~354 tokens available for the article
- ~3–5 tokens for the label (`" World"`, `" Sci/Tech"`, etc.)
- Very few articles get truncated at 384 tokens


In [5]:

# Label mapping and prompt templates.
ID2LABEL = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech",
}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
ALLOWED_LABELS = list(LABEL2ID.keys())


def build_prompt(article_text: str) -> str:
    return (
        "Classify the following news article into exactly one label: "
        "World, Sports, Business, or Sci/Tech.\n"
        "Return only the label.\n\n"
        f"Article: {article_text}\n"
        "Label:"
    )


def normalize_predicted_label(raw_text: str) -> str:
    text = raw_text.strip().lower()
    text = re.sub(r"[^a-z/ ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    # Direct exact matches first.
    direct = {
        "world": "World",
        "sports": "Sports",
        "business": "Business",
        "sci/tech": "Sci/Tech",
        "science/tech": "Sci/Tech",
        "science and technology": "Sci/Tech",
        "science technology": "Sci/Tech",
        "technology": "Sci/Tech",
        "science": "Sci/Tech",
    }
    if text in direct:
        return direct[text]

    # Substring fallback.
    if "sport" in text:
        return "Sports"
    if "business" in text or "finance" in text:
        return "Business"
    if "world" in text:
        return "World"
    if "sci" in text or "tech" in text or "science" in text or "technology" in text:
        return "Sci/Tech"

    return "UNKNOWN"


print(build_prompt(train_ds[0]["text"]))
print("Target label:", ID2LABEL[int(train_ds[0]["label"])])


Classify the following news article into exactly one label: World, Sports, Business, or Sci/Tech.
Return only the label.

Article: Congress closes in on reviving Net tax ban WASHINGTON - Lawmakers revived a long-stalled effort Wednesday to block state and local governments from taxing Internet connections, in a last-minute agreement to ban the taxes until 2007.
Label:
Target label: Business


In [6]:

# Load tokenizer once. We also ensure a valid pad token for batching.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def encode_supervised_example(example: dict) -> dict:
    prompt = build_prompt(example["text"])
    target = " " + ID2LABEL[int(example["label"])] + tokenizer.eos_token

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    target_ids = tokenizer(target, add_special_tokens=False)["input_ids"]

    # Keep all target tokens; truncate prompt if needed.
    max_prompt_len = max(1, MAX_SEQ_LENGTH - len(target_ids))
    prompt_ids = prompt_ids[:max_prompt_len]

    input_ids = prompt_ids + target_ids
    attention_mask = [1] * len(input_ids)
    labels = [-100] * len(prompt_ids) + target_ids

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


# Tokenize train/validation for supervised fine-tuning.
train_tokenized = train_ds.map(
    encode_supervised_example,
    remove_columns=train_ds.column_names,
    desc="Tokenizing train split",
)
val_tokenized = val_ds.map(
    encode_supervised_example,
    remove_columns=val_ds.column_names,
    desc="Tokenizing validation split",
)


def causal_lm_collator(features: list[dict]) -> dict:
    # Pad input_ids and attention_mask with tokenizer rules.
    batch = tokenizer.pad(
        {
            "input_ids": [f["input_ids"] for f in features],
            "attention_mask": [f["attention_mask"] for f in features],
        },
        padding=True,
        return_tensors="pt",
    )

    # Pad labels manually with -100 so prompt/pad tokens are ignored in loss.
    max_len = batch["input_ids"].shape[1]
    padded_labels = []
    for f in features:
        lbl = f["labels"]
        padded_labels.append(lbl + [-100] * (max_len - len(lbl)))
    batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)
    return batch


print(train_tokenized[0].keys())
print({k: len(v) for k, v in train_tokenized[0].items()})


config.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/17.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.15M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

Tokenizing train split:   0%|          | 0/110000 [00:00<?, ? examples/s]

Tokenizing validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask', 'labels'])
{'input_ids': 79, 'attention_mask': 79, 'labels': 79}



## Step 3: Model Architecture and Loading Helpers

### Granite 4.1 3B Base Architecture

`ibm-granite/granite-4.1-3b-base` is a decoder-only transformer with:

| Property | Value |
|----------|-------|
| Parameters | ~3.4 B |
| Architecture | Granite Transformer (LLaMA-family) |
| Layers | 32 transformer blocks |
| Hidden size | 3072 |
| Attention heads | 24 |
| Context length | 128K tokens |
| Vocab size | 49,152 |
| Training dtype | bfloat16 |

Each transformer block contains:
1. **Self-attention**: Q/K/V/O linear projections — these are the primary LoRA targets
2. **MLP (SiLU gate + up + down projections)**: also targeted by `"all-linear"`
3. **RMS LayerNorm**: not targeted by LoRA (no meaningful weight to adapt)

### Loading Strategies

| Strategy | `quantized_4bit` | `for_training` | Use Case |
|----------|------------------|----------------|----------|
| Inference bf16 | False | False | Base model eval |
| Training bf16 | False | True | LoRA fine-tuning |
| Inference 4-bit | True | False | QLoRA inference |
| Training 4-bit | True | True | QLoRA fine-tuning |

### Evaluation Strategy: Generation-Based Classification

We evaluate by generating label tokens (greedy decoding) and parsing the output:
```
prompt → model.generate() → " Business\n" → normalize → "Business" → label_id=2
```

This is the most faithful evaluation for a generative fine-tuning setup. The alternative (extracting logits over label tokens) would be faster but wouldn't measure the same skill the model is being trained to perform.

### Metrics

| Metric | Description |
|--------|-------------|
| `accuracy_strict` | All samples; UNKNOWN = wrong prediction |
| `macro_f1_known_only` | Macro-averaged F1 over parseable predictions only |
| `label_parse_coverage` | Fraction of predictions successfully parsed to a label |


In [ ]:

memory_log = []


def get_model_device(model: torch.nn.Module) -> torch.device:
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device("cpu")


def load_base_model(quantized_4bit: bool = False, for_training: bool = False) -> torch.nn.Module:
    """Load Granite 4.1 3B, optionally in 4-bit NF4 (QLoRA) mode.

    Args:
        quantized_4bit: If True, load with BitsAndBytes NF4 4-bit quantization (QLoRA).
        for_training: If True, enable gradient checkpointing and pin to GPU 0.

    Returns:
        The loaded model on the appropriate device.
    """
    if quantized_4bit:
        if not torch.cuda.is_available():
            raise RuntimeError("QLoRA requires CUDA. No GPU detected.")

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",          # Normal Float 4 — best quality for LLMs
            bnb_4bit_use_double_quant=True,      # Quantize the quantization constants too
            bnb_4bit_compute_dtype=get_compute_dtype(),
        )
        q_device_map = {"": 0} if for_training else "auto"
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map=q_device_map,
            trust_remote_code=True,
        )
    else:
        kwargs = {"trust_remote_code": True}
        if torch.cuda.is_available():
            kwargs["dtype"] = get_compute_dtype()   # transformers 5.x: 'dtype' replaces 'torch_dtype'
            if for_training:
                kwargs["device_map"] = {"": 0}
                kwargs["low_cpu_mem_usage"] = False
            else:
                kwargs["device_map"] = "auto"
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs)

    # Gradient checkpointing: recompute activations on the backward pass instead of storing
    # them, trading compute for memory. Required to fit a 3B model on 8 GB VRAM.
    if for_training:
        if hasattr(model, "gradient_checkpointing_enable"):
            model.gradient_checkpointing_enable()
        if hasattr(model, "config"):
            model.config.use_cache = False  # incompatible with gradient checkpointing

    return model


def predict_label(model: torch.nn.Module, article_text: str) -> tuple[str, str]:
    """Generate a label for one article using greedy decoding."""
    prompt = build_prompt(article_text)
    inputs = tokenizer(prompt, return_tensors="pt")

    device = get_model_device(model)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS_LABEL,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    label = normalize_predicted_label(generated_text)
    return label, generated_text


def evaluate_generation_classifier(
    model: torch.nn.Module,
    dataset: Dataset,
    run_name: str,
    max_samples: int | None = None,
) -> tuple[dict, pd.DataFrame]:
    """Evaluate a causal LM as a classifier by generating label tokens and parsing output.

    Uses greedy decoding per sample (no batching) to keep the evaluation loop simple.
    Reports both strict accuracy (UNKNOWN = wrong) and macro-F1 on parseable predictions.
    """
    n = len(dataset) if max_samples is None else min(max_samples, len(dataset))
    subset = dataset.select(range(n))

    rows = []
    y_true = []
    y_pred = []

    for ex in tqdm(subset, total=n, desc=f"{run_name} generation eval"):
        pred_label, raw_text = predict_label(model, ex["text"])
        true_label = ID2LABEL[int(ex["label"])]

        y_true.append(int(ex["label"]))
        y_pred.append(LABEL2ID.get(pred_label, -1))
        rows.append(
            {
                "text": ex["text"],
                "true_label": true_label,
                "pred_label": pred_label,
                "raw_generation": raw_text,
            }
        )

    # Strict accuracy counts UNKNOWN as wrong (fair baseline penalty).
    strict_accuracy = float(sum(int(p == t) for p, t in zip(y_pred, y_true)) / len(y_true))

    known_mask = [p != -1 for p in y_pred]
    if any(known_mask):
        y_true_known = [t for t, m in zip(y_true, known_mask) if m]
        y_pred_known = [p for p, m in zip(y_pred, known_mask) if m]
        macro_f1_known = float(f1_score(y_true_known, y_pred_known, average="macro"))
    else:
        macro_f1_known = 0.0

    parse_coverage = float(sum(known_mask) / len(known_mask))

    metrics = {
        "run_name": run_name,
        "samples_evaluated": n,
        "accuracy_strict": strict_accuracy,
        "macro_f1_known_only": macro_f1_known,
        "label_parse_coverage": parse_coverage,
    }

    return metrics, pd.DataFrame(rows)



## Step 4: Baseline Evaluation (Zero-Shot, Before Fine-Tuning)

### Purpose

The base evaluation answers: **"How well does the pre-trained model already perform at this task?"**

This serves as:
1. The **performance floor** — fine-tuning should meaningfully exceed this
2. A **sanity check** — if the base model gets 0% accuracy, check prompt and parser logic
3. A **comparison anchor** — LoRA and QLoRA gains are measured relative to this

### What We Expect

`granite-4.1-3b-base` is a strong pre-trained model. AG News classification with a clear instruction prompt should yield:
- **~80–95% accuracy** in zero-shot (the model understands the label names well)
- **~100% parse coverage** (the model generates recognizable label text)

If accuracy is lower (e.g., <50%), it would indicate prompt or tokenization issues.

### Memory and Execution Notes

- The base model is loaded in **bfloat16** (non-quantized) for the cleanest baseline
- Evaluation runs sample-by-sample (no batching) — expected time: ~10–30 min on 8 GB GPU
- After this cell, `base_model` is freed to reclaim VRAM before LoRA training


In [8]:

base_model = None
base_metrics = None
base_predictions = None

if should_run_stage("base"):
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    memory_log.append(memory_snapshot("before_base_load"))

    base_model = load_base_model(quantized_4bit=False, for_training=False)
    base_model.eval()

    memory_log.append(memory_snapshot("after_base_load"))

    base_metrics, base_predictions = evaluate_generation_classifier(
        base_model,
        gen_eval_ds,
        run_name="Base model",
        max_samples=GEN_EVAL_MAX_SAMPLES,
    )

    memory_log.append(memory_snapshot("after_base_eval"))

    print(base_metrics)
    base_predictions.head(10)
else:
    print(f"Base stage skipped for RUN_STAGE={RUN_STAGE}.")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/29.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Base model generation eval:   0%|          | 0/2000 [00:00<?, ?it/s]

{'run_name': 'Base model', 'samples_evaluated': 2000, 'accuracy_strict': 0.8915, 'macro_f1_known_only': 0.8895918817394437, 'label_parse_coverage': 1.0}


,text,true_label,pred_label,raw_generation
0,Fears for T N pension after talks Unions repre...,Business,Business,Business
1,The Race is On: Second Private Team Sets Launc...,Sci/Tech,Sci/Tech,Sci/Tech
2,Ky. Company Wins Grant to Study Peptides (AP) ...,Sci/Tech,Sci/Tech,Sci/Tech
3,Prediction Unit Helps Forecast Wildfires (AP) ...,Sci/Tech,Sci/Tech,Sci/Tech
4,Calif. Aims to Limit Farm-Related Smog (AP) AP...,Sci/Tech,Sci/Tech,Sci/Tech
5,Open Letter Against British Copyright Indoctri...,Sci/Tech,Sci/Tech,Sci/Tech
6,"Loosing the War on Terrorism \\""Sven Jaschan, ...",Sci/Tech,Sci/Tech,Sci/Tech
7,"FOAFKey: FOAF, PGP, Key Distribution, and Bloo...",Sci/Tech,Sci/Tech,Sci/Tech
8,E-mail scam targets police chief Wiltshire Pol...,Sci/Tech,Sci/Tech,Sci/Tech
9,"Card fraud unit nets 36,000 cards In its first...",Sci/Tech,Business,Business



## Step 5: LoRA Fine-Tuning

### LoRA Configuration

This notebook uses `r=16, lora_alpha=32` with `target_modules="all-linear"`. Here's what each parameter means:

| Parameter | Value | Effect |
|-----------|-------|--------|
| `r` | 16 | Rank — controls adapter width. More = more expressive, more parameters |
| `lora_alpha` | 32 | Scaling: effective lr multiplier = `alpha/r = 2.0` for the adapter |
| `lora_dropout` | 0.05 | Light regularization on adapter activations |
| `target_modules` | `"all-linear"` | Applies adapters to every `nn.Linear` in the model |
| `bias` | `"none"` | Don't train bias parameters — saves memory with minimal accuracy cost |

**Parameter count**: `r=16` with `"all-linear"` on a 3B model → ~31M trainable / 3.4B total ≈ **0.9%**.

### Training Configuration

| Hyperparameter | Value | Rationale |
|----------------|-------|-----------|
| `num_train_epochs` | 1 | One pass through 110K samples is sufficient for strong adaptation |
| `learning_rate` | 2e-4 | Standard for LoRA; higher than full fine-tuning since adapters start from zero |
| `per_device_batch_size` | 1 | Tight VRAM — effective batch = 1×16 = 16 via gradient accumulation |
| `gradient_accumulation_steps` | 16 | Simulates batch=16 without storing 16 samples in VRAM at once |
| `warmup_ratio` | 0.03 | Warmup for the first 3% of steps to stabilize adapter initialization |
| `weight_decay` | 0.01 | Mild L2 regularization |
| `gradient_checkpointing` | True | Trade ~30% compute for ~40% VRAM reduction on activations |
| `bf16` | True | Mixed-precision training — reduces memory and speeds up computation |

### Memory Optimization Techniques

Training a 3B model on 8 GB VRAM requires all of these simultaneously:

1. **LoRA (frozen base)**: eliminates gradients and optimizer states for 99.1% of parameters
2. **Gradient checkpointing**: frees activation memory between forward/backward passes
3. **BF16 mixed precision**: halves weight and activation memory vs FP32
4. **Gradient accumulation**: uses batch=1 in memory, accumulates 16 steps for effective batch=16
5. **`use_cache=False`**: disables the KV cache (incompatible with gradient checkpointing)

Without these, training would require ~28 GB VRAM. Together, they compress the requirement to ~7 GB.

### Training Process

The `Trainer` handles:
1. Forward pass through frozen base + active adapters
2. Loss computed only on label tokens (masked prompt with `-100`)
3. Backward pass through adapters only (base is `requires_grad=False`)
4. Gradient accumulation for 16 steps
5. AdamW optimizer step on adapter parameters
6. Checkpoint saved at epoch end

Expected training time: ~30–90 minutes for 1 epoch on RTX 4060.


In [9]:

# Build TrainingArguments in a version-robust way (Transformers may use eval_strategy or evaluation_strategy).
TRAINING_ARGS_PARAMS = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
EVAL_STRATEGY_KEY = "eval_strategy" if "eval_strategy" in TRAINING_ARGS_PARAMS else "evaluation_strategy"


def make_training_args(output_dir: Path, run_name: str) -> TrainingArguments:
    args_dict = {
        "output_dir": str(output_dir),
        "overwrite_output_dir": True,
        "num_train_epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "per_device_train_batch_size": PER_DEVICE_BATCH_SIZE,
        "per_device_eval_batch_size": PER_DEVICE_BATCH_SIZE,
        "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
        "logging_steps": LOGGING_STEPS,
        "save_strategy": "epoch",
        "load_best_model_at_end": False,
        "report_to": "none",
        "run_name": run_name,
        "remove_unused_columns": False,
        "dataloader_num_workers": 2,
        "gradient_checkpointing": True,
    }
    args_dict[EVAL_STRATEGY_KEY] = "epoch"

    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        args_dict["bf16"] = True
    elif torch.cuda.is_available():
        args_dict["fp16"] = True

    supported_args = {k: v for k, v in args_dict.items() if k in TRAINING_ARGS_PARAMS}
    return TrainingArguments(**supported_args)


def print_trainable_ratio(model: torch.nn.Module) -> None:
    trainable = 0
    total = 0
    for p in model.parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
    pct = 100 * trainable / total
    print(f"Trainable params: {trainable:,}")
    print(f"Total params: {total:,}")
    print(f"Trainable ratio: {pct:.4f}%")


In [ ]:

# Free baseline model before LoRA stage.
# IMPORTANT: set base_model = None in notebook scope BEFORE calling cleanup so Python's
# refcount drops to zero and the CUDA allocator can actually reclaim the memory.
base_model = None
cleanup_torch_objects()

lora_stage_ok = False
lora_stage_error = None
lora_base_model = None
lora_model = None
lora_trainer = None
lora_train_result = None
lora_eval_loss = None
lora_logs_df = pd.DataFrame()
lora_metrics = None
lora_predictions = None

if should_run_stage("lora"):
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        free_vram = torch.cuda.mem_get_info()[0] / 1e9
        print(f"Free VRAM before LoRA load: {free_vram:.2f} GB")

    memory_log.append(memory_snapshot("before_lora_load"))

    try:
        lora_base_model = load_base_model(quantized_4bit=False, for_training=True)

        lora_config = LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            target_modules="all-linear",
            bias="none",
            task_type=TaskType.CAUSAL_LM,
        )

        lora_model = get_peft_model(lora_base_model, lora_config)
        print_trainable_ratio(lora_model)

        memory_log.append(memory_snapshot("after_lora_model_prepare"))

        lora_train_args = make_training_args(ARTIFACTS_DIR / "lora_run", run_name="granite41-lora")
        lora_trainer = Trainer(
            model=lora_model,
            args=lora_train_args,
            train_dataset=train_tokenized,
            eval_dataset=val_tokenized,
            data_collator=causal_lm_collator,
        )

        lora_train_result = lora_trainer.train()
        lora_eval_loss = lora_trainer.evaluate()

        # Save LoRA adapter + tokenizer.
        lora_trainer.model.save_pretrained(LORA_ADAPTER_DIR)
        tokenizer.save_pretrained(LORA_ADAPTER_DIR)

        memory_log.append(memory_snapshot("after_lora_train_save"))

        lora_logs_df = pd.DataFrame(lora_trainer.state.log_history)
        print("LoRA train metrics:")
        print(lora_train_result.metrics)
        print("LoRA token-level eval:")
        print(lora_eval_loss)

        lora_stage_ok = True

    except Exception as e:
        lora_stage_error = e
        lora_trainer = None
        lora_model = None
        lora_base_model = None
        cleanup_torch_objects()
        if FAIL_ON_STAGE_ERROR:
            raise
        print("LoRA stage failed:")
        print(e)
else:
    print(f"LoRA stage skipped for RUN_STAGE={RUN_STAGE}.")

if not lora_logs_df.empty:
    lora_logs_df.tail(15)


In [ ]:

# Generation-based evaluation for LoRA model.
if should_run_stage("lora") and lora_stage_ok and lora_model is not None:
    lora_model.eval()

    lora_metrics, lora_predictions = evaluate_generation_classifier(
        lora_model,
        gen_eval_ds,
        run_name="LoRA",
        max_samples=GEN_EVAL_MAX_SAMPLES,
    )

    memory_log.append(memory_snapshot("after_lora_generation_eval"))

    print(lora_metrics)
    lora_predictions.head(10)
elif should_run_stage("lora"):
    print("LoRA generation eval skipped because LoRA training did not complete.")
else:
    print(f"LoRA generation eval skipped for RUN_STAGE={RUN_STAGE}.")



## Step 6: QLoRA Fine-Tuning (4-bit NF4 Base + LoRA Adapters)

### What Changes vs LoRA

QLoRA uses the exact same adapter architecture as LoRA. The only difference is how the **base model is loaded**:

| Aspect | LoRA | QLoRA |
|--------|------|-------|
| Base model storage | bfloat16 (~6 GB) | 4-bit NF4 (~1.7 GB) |
| Base model compute | bfloat16 | Dequantized to bf16 per operation |
| Adapter dtype | bfloat16 | bfloat16 (unchanged) |
| Peak VRAM | ~7–7.5 GB | ~4–5 GB |
| Training speed | Faster | ~10–30% slower (dequant overhead) |

### BitsAndBytes NF4 Configuration

```python
BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # Normal Float 4 — optimal for pretrained LLMs
    bnb_4bit_use_double_quant=True,      # Compress quantization constants too
    bnb_4bit_compute_dtype=bfloat16,     # Dequantize to bf16 for compute
)
```

### `prepare_model_for_kbit_training`

After loading a 4-bit quantized model, `prepare_model_for_kbit_training()` is required. It:
1. Converts any `float32` layer norms to `bfloat16` (avoids dtype mismatch)
2. Sets the model's internal training mode for quantized operations
3. Configures gradient hooks to work with quantized tensors

Without this step, training will fail with dtype errors.

### Memory Budget for QLoRA on 8 GB

| Component | Memory |
|-----------|--------|
| 4-bit base model | ~1.7 GB |
| Adapter params (bf16) | ~0.24 GB |
| Optimizer states (adapters only) | ~0.48 GB |
| Activations + overhead | ~2–3 GB |
| **Total** | **~4.5–5.5 GB** |

This leaves ~2.5–3.5 GB headroom on an 8 GB GPU — comfortable for batch=1.

### When This Stage is Skipped

If `CUDA` or `bitsandbytes` are unavailable, the QLoRA cell raises a `RuntimeError` immediately (fail-fast). On CPU-only machines, you can still run Base and LoRA stages by setting `FAIL_ON_STAGE_ERROR=false` in the environment.


In [ ]:

# Release LoRA trainer objects before QLoRA stage.
# Must nullify notebook-scope references before gc.collect so refcounts drop to zero.
lora_trainer = None
lora_model = None
lora_base_model = None
cleanup_torch_objects()

qlora_metrics = None
qlora_predictions = None
qlora_logs_df = pd.DataFrame()
qlora_train_result = None
qlora_eval_loss = None
qlora_base_model = None
qlora_model = None
qlora_trainer = None

can_run_qlora = runtime.get("qlora_supported", False)

if should_run_stage("qlora"):
    if not can_run_qlora:
        raise RuntimeError("QLoRA stage requested, but CUDA and/or bitsandbytes support was not detected.")

    try:
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
            free_vram = torch.cuda.mem_get_info()[0] / 1e9
            print(f"Free VRAM before QLoRA load: {free_vram:.2f} GB")

        memory_log.append(memory_snapshot("before_qlora_load"))

        qlora_base_model = load_base_model(quantized_4bit=True, for_training=True)
        qlora_base_model = prepare_model_for_kbit_training(qlora_base_model)

        qlora_config = LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            target_modules="all-linear",
            bias="none",
            task_type=TaskType.CAUSAL_LM,
        )

        qlora_model = get_peft_model(qlora_base_model, qlora_config)
        print_trainable_ratio(qlora_model)

        memory_log.append(memory_snapshot("after_qlora_model_prepare"))

        qlora_train_args = make_training_args(ARTIFACTS_DIR / "qlora_run", run_name="granite41-qlora")
        qlora_trainer = Trainer(
            model=qlora_model,
            args=qlora_train_args,
            train_dataset=train_tokenized,
            eval_dataset=val_tokenized,
            data_collator=causal_lm_collator,
        )

        qlora_train_result = qlora_trainer.train()
        qlora_eval_loss = qlora_trainer.evaluate()

        # Save QLoRA adapter + tokenizer.
        qlora_trainer.model.save_pretrained(QLORA_ADAPTER_DIR)
        tokenizer.save_pretrained(QLORA_ADAPTER_DIR)

        memory_log.append(memory_snapshot("after_qlora_train_save"))

        qlora_logs_df = pd.DataFrame(qlora_trainer.state.log_history)

        qlora_model.eval()
        qlora_metrics, qlora_predictions = evaluate_generation_classifier(
            qlora_model,
            gen_eval_ds,
            run_name="QLoRA",
            max_samples=GEN_EVAL_MAX_SAMPLES,
        )

        memory_log.append(memory_snapshot("after_qlora_generation_eval"))

        print("QLoRA train metrics:")
        print(qlora_train_result.metrics)
        print("QLoRA token-level eval:")
        print(qlora_eval_loss)
        print("QLoRA generation metrics:")
        print(qlora_metrics)

    except Exception as e:
        qlora_trainer = None
        qlora_model = None
        qlora_base_model = None
        cleanup_torch_objects()
        if FAIL_ON_STAGE_ERROR:
            raise
        print("QLoRA stage failed:")
        print(e)
        can_run_qlora = False
else:
    print(f"QLoRA stage skipped for RUN_STAGE={RUN_STAGE}.")


In [ ]:

if should_run_stage("qlora") and not qlora_logs_df.empty:
    qlora_logs_df.tail(15)
elif should_run_stage("qlora"):
    print("No QLoRA training logs available.")
else:
    print(f"QLoRA logs skipped for RUN_STAGE={RUN_STAGE}.")



## Step 7: Results Analysis — Base vs LoRA vs QLoRA

### Metrics Explained

| Metric | Interpretation |
|--------|----------------|
| `accuracy_strict` | What fraction of 2000 articles did the model label correctly? UNKNOWN counts as wrong. |
| `macro_f1_known_only` | Average F1 over parseable predictions (equal weight per class). Identifies if one class is driving accuracy. |
| `label_parse_coverage` | Did the model output a recognizable label word? Base model should be ~100%; if LoRA drops this, check the prompt. |
| `train_runtime_sec` | Wall-clock training time in seconds |
| `token_eval_loss` | Perplexity-scale loss on label tokens in the validation set (lower = better) |

### Expected Outcome Pattern

Typical results on AG News with this setup:

| Stage | Accuracy (typical range) |
|-------|--------------------------|
| Base (zero-shot) | 80–92% |
| LoRA (1 epoch) | 92–97% |
| QLoRA (1 epoch) | 90–96% |

**LoRA** typically exceeds base by a meaningful margin after just 1 epoch.  
**QLoRA** should match LoRA closely, with ≤2% accuracy gap from quantization overhead.  
If QLoRA is significantly worse, consider: higher `r`, more epochs, or checking for NaN loss during training.

### Interpreting the Memory Profile

The `memory_df` below shows GPU allocation at each checkpoint:
- `before_*_load`: baseline before model load
- `after_*_load`: memory footprint of the loaded model
- `after_*_train_save`: peak after training + optimizer state + adapter save
- `after_*_generation_eval`: eval footprint (inference only, no optimizer)

**Key signal**: `gpu_max_allocated_gb` shows the peak that was actually reached — this is what matters for OOM risk.


In [ ]:

comparison_rows = []
comparison_df = pd.DataFrame()

if should_run_stage("final"):
    if base_metrics is None:
        print("Final stage: computing base generation metrics for comparison.")
        base_eval_model = load_base_model(quantized_4bit=False, for_training=False)
        base_eval_model.eval()
        base_metrics, _ = evaluate_generation_classifier(
            base_eval_model,
            gen_eval_ds,
            run_name="Base model (final)",
            max_samples=GEN_EVAL_MAX_SAMPLES,
        )
        cleanup_torch_objects(base_eval_model)

    if lora_metrics is None and LORA_ADAPTER_DIR.exists() and any(LORA_ADAPTER_DIR.iterdir()):
        print("Final stage: computing LoRA generation metrics from saved adapter.")
        lora_eval_base = load_base_model(quantized_4bit=False, for_training=False)
        lora_eval_model = PeftModel.from_pretrained(lora_eval_base, str(LORA_ADAPTER_DIR))
        lora_eval_model.eval()
        lora_metrics, _ = evaluate_generation_classifier(
            lora_eval_model,
            gen_eval_ds,
            run_name="LoRA (reloaded final)",
            max_samples=GEN_EVAL_MAX_SAMPLES,
        )
        cleanup_torch_objects(lora_eval_model, lora_eval_base)

    if qlora_metrics is None and runtime.get("qlora_supported", False) and QLORA_ADAPTER_DIR.exists() and any(QLORA_ADAPTER_DIR.iterdir()):
        print("Final stage: computing QLoRA generation metrics from saved adapter.")
        qlora_eval_base = load_base_model(quantized_4bit=True, for_training=False)
        qlora_eval_model = PeftModel.from_pretrained(qlora_eval_base, str(QLORA_ADAPTER_DIR))
        qlora_eval_model.eval()
        qlora_metrics, _ = evaluate_generation_classifier(
            qlora_eval_model,
            gen_eval_ds,
            run_name="QLoRA (reloaded final)",
            max_samples=GEN_EVAL_MAX_SAMPLES,
        )
        cleanup_torch_objects(qlora_eval_model, qlora_eval_base)

    comparison_rows.append(
        {
            "run": "Base",
            "accuracy_strict": base_metrics["accuracy_strict"] if base_metrics else np.nan,
            "macro_f1_known_only": base_metrics["macro_f1_known_only"] if base_metrics else np.nan,
            "label_parse_coverage": base_metrics["label_parse_coverage"] if base_metrics else np.nan,
            "train_runtime_sec": np.nan,
            "token_eval_loss": np.nan,
        }
    )

    comparison_rows.append(
        {
            "run": "LoRA",
            "accuracy_strict": lora_metrics["accuracy_strict"] if lora_metrics else np.nan,
            "macro_f1_known_only": lora_metrics["macro_f1_known_only"] if lora_metrics else np.nan,
            "label_parse_coverage": lora_metrics["label_parse_coverage"] if lora_metrics else np.nan,
            "train_runtime_sec": lora_train_result.metrics.get("train_runtime", np.nan) if lora_train_result else np.nan,
            "token_eval_loss": lora_eval_loss.get("eval_loss", np.nan) if lora_eval_loss else np.nan,
        }
    )

    comparison_rows.append(
        {
            "run": "QLoRA",
            "accuracy_strict": qlora_metrics["accuracy_strict"] if qlora_metrics else np.nan,
            "macro_f1_known_only": qlora_metrics["macro_f1_known_only"] if qlora_metrics else np.nan,
            "label_parse_coverage": qlora_metrics["label_parse_coverage"] if qlora_metrics else np.nan,
            "train_runtime_sec": qlora_train_result.metrics.get("train_runtime", np.nan) if qlora_train_result else np.nan,
            "token_eval_loss": qlora_eval_loss.get("eval_loss", np.nan) if qlora_eval_loss else np.nan,
        }
    )

    comparison_df = pd.DataFrame(comparison_rows)
    comparison_df
else:
    print(f"Comparison stage skipped for RUN_STAGE={RUN_STAGE}.")


In [ ]:

# Memory usage snapshots across baseline/LoRA/QLoRA pipeline.
memory_df = pd.DataFrame(memory_log)
memory_df



## Step 8: Reload Saved Adapters and Run Inference

### Deployment-Style Adapter Loading

This section simulates how you would use a trained adapter in production:

1. **Load** the base model fresh (no training state)
2. **Attach** the saved LoRA adapter using `PeftModel.from_pretrained()`
3. **Run** inference — no training artifacts, clean eval mode

This is the correct deployment pattern: the base model stays on disk/registry, and adapters are lightweight add-ons (typically 50–200 MB for a 3B model).

### Adapter Files Saved

After training, the adapter directory contains:
```
lora_adapter/
├── adapter_config.json      # LoRA hyperparameters (r, alpha, target_modules, etc.)
├── adapter_model.safetensors # Adapter weight matrices (A, B for each layer)
├── tokenizer.json           # Tokenizer vocabulary
├── tokenizer_config.json    # Tokenizer settings
└── special_tokens_map.json  # Special token definitions
```

The adapter checkpoint is typically **50–200 MB** vs the ~6 GB base model — this is the storage efficiency win of PEFT.

### What to Look For in the Inference Output

- **Base model**: Should match the zero-shot baseline accuracy
- **LoRA reloaded**: Should match (or very closely approximate) the LoRA training evaluation result
- **QLoRA reloaded**: Should match the QLoRA training evaluation result
- Any difference between in-training eval and reloaded eval indicates a checkpoint loading issue


In [ ]:

# Release any leftover training objects first.
cleanup_torch_objects()

inference_df = pd.DataFrame()

if should_run_stage("final"):
    inference_examples = [
        gen_eval_ds[0]["text"],
        gen_eval_ds[1]["text"],
        gen_eval_ds[2]["text"],
    ]

    # Reload a clean base model for baseline inference.
    base_infer_model = load_base_model(quantized_4bit=False, for_training=False)
    base_infer_model.eval()

    # Reload another clean base model, then attach LoRA adapter when available.
    lora_reloaded_model = None
    if LORA_ADAPTER_DIR.exists() and any(LORA_ADAPTER_DIR.iterdir()):
        lora_base_for_reload = load_base_model(quantized_4bit=False, for_training=False)
        lora_reloaded_model = PeftModel.from_pretrained(lora_base_for_reload, str(LORA_ADAPTER_DIR))
        lora_reloaded_model.eval()
    else:
        print("LoRA adapter unavailable; skipping LoRA reload inference.")

    # Optionally reload QLoRA adapter using a 4-bit base model.
    qlora_reloaded_model = None
    if runtime.get("qlora_supported", False) and QLORA_ADAPTER_DIR.exists() and any(QLORA_ADAPTER_DIR.iterdir()):
        qlora_base_infer = load_base_model(quantized_4bit=True, for_training=False)
        qlora_reloaded_model = PeftModel.from_pretrained(qlora_base_infer, str(QLORA_ADAPTER_DIR))
        qlora_reloaded_model.eval()

    rows = []
    for text in inference_examples:
        base_label, base_raw = predict_label(base_infer_model, text)

        row = {
            "article": text[:220] + ("..." if len(text) > 220 else ""),
            "base_pred": base_label,
            "base_raw": base_raw,
        }

        if lora_reloaded_model is not None:
            lora_label, lora_raw = predict_label(lora_reloaded_model, text)
            row["lora_pred"] = lora_label
            row["lora_raw"] = lora_raw
        else:
            row["lora_pred"] = "SKIPPED"
            row["lora_raw"] = "LoRA adapter unavailable"

        if qlora_reloaded_model is not None:
            qlora_label, qlora_raw = predict_label(qlora_reloaded_model, text)
            row["qlora_pred"] = qlora_label
            row["qlora_raw"] = qlora_raw

        rows.append(row)

    inference_df = pd.DataFrame(rows)
    inference_df
else:
    print(f"Final inference stage skipped for RUN_STAGE={RUN_STAGE}.")



## Optional: Merge Adapter into Base Model for Simpler Inference

You can merge LoRA weights into the base model (one-time operation) to avoid loading adapter files separately:

```python
merged_model = lora_reloaded_model.merge_and_unload()
merged_model.save_pretrained("/path/to/merged-model")
```

Use this when deployment simplicity matters more than adapter modularity.



## Troubleshooting Guide

### OOM (Out of Memory) During Training

**Symptoms**: `CUDA out of memory` during the LoRA or QLoRA training cell.

**Solutions** (apply in order):
1. **Reduce `MAX_SEQ_LENGTH`**: try 256 or 192. Shorter sequences → smaller activations.
2. **Switch to QLoRA**: use `quantized_4bit=True` for the base model.
3. **Reduce `GRAD_ACCUM_STEPS`** (e.g., 8) — counterintuitively, fewer steps can reduce peak memory in some transformer implementations.
4. **Verify no other GPU processes**: run `nvidia-smi` and kill any other processes using VRAM.
5. **Clear GPU cache between stages**: the `cleanup_torch_objects()` calls handle this, but call it manually if needed.

### `TypeError: unexpected keyword argument 'overwrite_output_dir'`

This error means the `TrainingArguments` filter (`supported_args`) is not active. Ensure you are running the latest version of cell 16 (`make_training_args`), which filters unsupported arguments.

### `[transformers] 'torch_dtype' is deprecated`

Fixed in this notebook: `load_base_model` now uses `kwargs["dtype"]` (transformers 5.x API). If you see this in older checkpoints, update cell 12.

### LoRA Model Gives Low Accuracy After Reloading

**Check**:
1. `LORA_ADAPTER_DIR` path is correct and contains `adapter_model.safetensors`
2. The **same base model** (`MODEL_ID`) is used for both training and reloading
3. The tokenizer loaded alongside the adapter matches what was used during training

### QLoRA Training NaN Loss

**Cause**: dtype mismatch between 4-bit base and adapters.  
**Fix**: Ensure `prepare_model_for_kbit_training()` is called before `get_peft_model()`. Also ensure `bnb_4bit_compute_dtype` matches the adapter compute dtype (both should be `bfloat16`).

### `bitsandbytes` CUDA not found

**Cause**: `bitsandbytes` compiled against a different CUDA version.  
**Fix**:
```bash
pip install bitsandbytes --upgrade
# or for specific CUDA:
pip install bitsandbytes --index-url https://huggingface.github.io/bitsandbytes/
```

### Model Loads on CPU Instead of GPU

**Cause**: `device_map="auto"` selected CPU due to low free VRAM.  
**Fix**: Free VRAM before loading (close browser tabs, other processes), then reload. Check with `nvidia-smi`.

---

## Key Takeaways

You built a **complete, end-to-end PEFT pipeline** that runs on a single consumer GPU:

### What You Demonstrated

| Capability | How |
|------------|-----|
| Dataset preparation | HF Datasets with train/val/test splits, prompt masking |
| Zero-shot baseline | Generation-based classification with metric tracking |
| LoRA fine-tuning | `r=16` adapters on all linear layers, bf16, gradient checkpointing |
| QLoRA fine-tuning | 4-bit NF4 base + same LoRA adapters via BitsAndBytes |
| Quantitative comparison | Strict accuracy, macro-F1, parse coverage, memory profiles |
| Adapter save/reload | `save_pretrained` + `PeftModel.from_pretrained` deployment pattern |
| GPU memory profiling | `memory_snapshot` across all stages |

### Key Numbers to Remember

| Concept | This Run |
|---------|----------|
| Trainable params (LoRA) | ~31M / 3.4B total = **0.9%** |
| Base model VRAM (bf16) | ~6 GB |
| QLoRA base VRAM (4-bit) | ~1.7 GB |
| Adapter checkpoint size | ~50–200 MB |

### Next Steps

1. **Increase `r`** to 32 or 64 for more adapter capacity (2-4× more trainable params)
2. **More epochs**: 2–3 epochs often yield another 1–2% accuracy gain
3. **Try other datasets**: the same pipeline works for any text classification task
4. **Merge adapters**: `model.merge_and_unload()` creates a single merged model for simpler deployment
5. **Quantize the merged model**: apply GPTQ or AWQ to the merged model for production inference
6. **Explore other PEFT methods**: IA³, (IA)³, DoRA, LoftQ — compare with LoRA on the same task
